In [ ]:
import pandas as pd

df = pd.read_csv(
    r"D:\QCBM Molecular Augmentation Lite\data\raw\1Mstoned_vsc_initial_dataset_insilico_chemistry42_filtered.csv",
    nrows=5
)

print(df.shape)
print(df.columns.tolist())

df.head()

In [ ]:
import pandas as pd

df = pd.read_csv(
    r"D:\QCBM Molecular Augmentation Lite\data\raw\650known_inhibitors_smiles_and_selfies.csv",
    nrows=5
)

print(df.shape)
print(df.columns.tolist())

df.head()

In [ ]:
import pandas as pd

df = pd.read_csv(
    r"D:\QCBM Molecular Augmentation Lite\data\raw\KRAS_G12D_inhibitors_update202209_updated.csv",
    nrows=5
)

print(df.shape)
print(df.columns.tolist())

df.head()

In [ ]:
import pandas as pd

files = [
    r"D:\QCBM Molecular Augmentation Lite\data\raw\1Mstoned_vsc_initial_dataset_insilico_chemistry42_filtered.csv",
    r"D:\QCBM Molecular Augmentation Lite\data\raw\650known_inhibitors_smiles_and_selfies.csv",
    r"D:\QCBM Molecular Augmentation Lite\data\raw\KRAS_G12D_inhibitors_update202209_updated.csv"
]

for f in files:
    
    df = pd.read_csv(f)

    print("=" * 80)
    print(f)
    print("Rows :", len(df))
    print("Cols :", len(df.columns))
    print(df.columns.tolist())

# QCBM Molecular Augmentation
## PySpark Big Data Processing Pipeline

Goals

- Load 1M molecule dataset
- Validate SMILES
- Remove duplicates
- Create benchmark datasets
- Generate descriptors
- Generate fingerprints
- Store efficient parquet files

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

import pandas as pd
import numpy as np

from rdkit import Chem

In [3]:
import os

os.environ["JAVA_HOME"] = r"C:\Users\debsh\Java\OpenJDK25U-jdk_x64_windows_hotspot_25.0.3_9\jdk-25.0.3+9"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = C:\Users\debsh\Java\OpenJDK25U-jdk_x64_windows_hotspot_25.0.3_9\jdk-25.0.3+9


# QCBM Molecular Augmentation

## Big Data Processing Pipeline (Dask)

This notebook performs:

- Raw dataset ingestion
- Data validation
- SMILES cleaning
- Deduplication
- Benchmark dataset preparation
- Parquet conversion
- Feature store generation
- Train / Validation / Test splits

Outputs are stored in:

data/processed/
data/benchmark_splits/
data/embeddings/

In [17]:
import warnings

warnings.filterwarnings("ignore")

import os

import numpy as np
import pandas as pd

import dask.dataframe as dd

from rdkit import Chem


In [18]:
PROJECT_ROOT = r"D:\QCBM Molecular Augmentation Lite"

RAW_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "raw"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

EMBEDDINGS_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "embeddings"
)

BENCHMARK_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "benchmark_splits"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
os.makedirs(BENCHMARK_DIR, exist_ok=True)

In [19]:
STONED_PATH = os.path.join(
    RAW_DIR,
    "1Mstoned_vsc_initial_dataset_insilico_chemistry42_filtered.csv"
)

KNOWN_PATH = os.path.join(
    RAW_DIR,
    "650known_inhibitors_smiles_and_selfies.csv"
)

KRAS_PATH = os.path.join(
    RAW_DIR,
    "KRAS_G12D_inhibitors_update202209_updated.csv"
)

In [20]:
stoned_ddf = dd.read_csv(
    STONED_PATH
)

known_df = pd.read_csv(
    KNOWN_PATH
)

kras_df = pd.read_csv(
    KRAS_PATH
)

print("STONED PARTITIONS:", stoned_ddf.npartitions)

print("KNOWN:", known_df.shape)

print("KRAS:", kras_df.shape)

STONED PARTITIONS: 1
KNOWN: (636, 6)
KRAS: (645, 5)


In [21]:
stoned_ddf.head()

,Unnamed: 0,smiles
0,0,CC1=C(CNc2cccnc2C)CN(C)C1OCCCCCN
1,1,CCN1CCCC1Cc1nc2c(c(N(C)CCCNC)n1)CCN(c1cncc3ccc...
2,2,C1=CCC(CCOc2nc3c([nH]2)CCN(c2ccccc2)C3)CC1
3,3,C=CC1(Oc2nc(N3CCN(CC)CC3)c3cnc(-c4ccccc4NC)c(F...
4,4,C=CNC(C)CCOc1nc(CN2CCNCC2)c2cnc(-c3ccccc3C(F)(...


In [22]:
known_df.head()

,id,smiles,KRAS G12D SPR KD (nM),KRAS G12D binding IC50 (nM),pERK IC50 (nM),selfies
0,1,CN1[C@H](COc2nc3cc(-c4cc(O)cc5ccccc45)ncc3c(N3...,97.7,124.7,3159.1,[C][N][C@H1][Branch2][Branch1][=Branch1][C][O]...
1,2,CN1[C@H](COc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)c...,2.4,2.7,721.4,[C][N][C@H1][Branch2][Branch1][#Branch2][C][O]...
2,3,Cn1c(CCOc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)cc4c...,8.3,9.5,10283.1,[C][N][C][Branch2][Branch1][O][C][C][O][C][=N]...
3,4,Oc1cc2ccccc2c(-c(ncc(c2n3)c(N4CC(CC5)NC5C4)nc3...,155.7,496.2,8530,[O][C][=C][C][=C][C][=C][C][=C][Ring1][=Branch...
4,5,Cn1nccc1COc1nc(N2CC(CC3)NC3C2)c(cnc(-c2cc(O)cc...,294.8,722.9,8193.8,[C][N][N][=C][C][=C][Ring1][Branch1][C][O][C][...


In [23]:
kras_df.head()

,id,smiles,KRAS G12D SPR KD (nM),KRAS G12D binding IC50 (nM),pERK IC50 (nM)
0,1,CN1[C@H](COc2nc3cc(-c4cc(O)cc5ccccc45)ncc3c(N3...,97.7,124.7,3159.1
1,2,CN1[C@H](COc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)c...,2.4,2.7,721.4
2,3,Cn1c(CCOc2nc(N3CC(CC4)NC4C3)c(cnc(-c3cc(O)cc4c...,8.3,9.5,10283.1
3,4,Oc1cc2ccccc2c(-c(ncc(c2n3)c(N4CC(CC5)NC5C4)nc3...,155.7,496.2,8530
4,5,Cn1nccc1COc1nc(N2CC(CC3)NC3C2)c(cnc(-c2cc(O)cc...,294.8,722.9,8193.8


In [24]:
print("STONED COLUMNS")

print(stoned_ddf.columns)

print("\nKNOWN COLUMNS")

print(known_df.columns)

print("\nKRAS COLUMNS")

print(kras_df.columns)

STONED COLUMNS
Index(['Unnamed: 0', 'smiles'], dtype='str')

KNOWN COLUMNS
Index(['id', 'smiles', 'KRAS G12D SPR KD  (nM)', 'KRAS G12D binding IC50 (nM)',
       'pERK IC50 (nM)', 'selfies'],
      dtype='str')

KRAS COLUMNS
Index(['id', 'smiles', 'KRAS G12D SPR KD  (nM)', 'KRAS G12D binding IC50 (nM)',
       'pERK IC50 (nM)'],
      dtype='str')


In [26]:
stoned_ddf = stoned_ddf.rename(
    columns={
        "Unnamed: 0": "id"
    }
)

stoned_ddf.head()

,id,smiles
0,0,CC1=C(CNc2cccnc2C)CN(C)C1OCCCCCN
1,1,CCN1CCCC1Cc1nc2c(c(N(C)CCCNC)n1)CCN(c1cncc3ccc...
2,2,C1=CCC(CCOc2nc3c([nH]2)CCN(c2ccccc2)C3)CC1
3,3,C=CC1(Oc2nc(N3CCN(CC)CC3)c3cnc(-c4ccccc4NC)c(F...
4,4,C=CNC(C)CCOc1nc(CN2CCNCC2)c2cnc(-c3ccccc3C(F)(...


In [27]:
def valid_smiles(
    smiles
):
    try:

        return (
            Chem.MolFromSmiles(
                str(smiles)
            )
            is not None
        )

    except Exception:

        return False

In [28]:
sample_df = stoned_ddf.head(
    10000
)

sample_df["valid"] = sample_df[
    "smiles"
].apply(
    valid_smiles
)

sample_df["valid"].value_counts()

[11:11:42] Explicit valence for atom # 20 P, 7, is greater than permitted


valid
True     9999
False       1
Name: count, dtype: int64

In [29]:
stoned_ddf = stoned_ddf.dropna(
    subset=["smiles"]
)

stoned_ddf

,id,smiles
npartitions=1,,
,int64,string
,...,...


In [31]:
before_count = (
    stoned_ddf.shape[0]
    .compute()
)

print(
    "Before:",
    before_count
)

Before: 1010510


In [32]:
stoned_ddf = (
    stoned_ddf
    .drop_duplicates(
        subset=["smiles"]
    )
)

In [33]:
after_count = (
    stoned_ddf.shape[0]
    .compute()
)

print(
    "After:",
    after_count
)

After: 1010510


In [34]:
print(
    "Removed:",
    before_count - after_count
)

Removed: 0


In [35]:
processed_stoned_path = os.path.join(
    PROCESSED_DIR,
    "stoned_1m.parquet"
)

stoned_ddf.to_parquet(
    processed_stoned_path,
    engine="pyarrow",
    overwrite=True
)

In [36]:
known_df.to_parquet(
    os.path.join(
        PROCESSED_DIR,
        "known_inhibitors.parquet"
    ),
    index=False
)

kras_df.to_parquet(
    os.path.join(
        PROCESSED_DIR,
        "kras_activity.parquet"
    ),
    index=False
)

In [37]:
processed_stoned = dd.read_parquet(
    processed_stoned_path
)

processed_stoned.head()

,id,smiles
0,0,CC1=C(CNc2cccnc2C)CN(C)C1OCCCCCN
1,1,CCN1CCCC1Cc1nc2c(c(N(C)CCCNC)n1)CCN(c1cncc3ccc...
2,2,C1=CCC(CCOc2nc3c([nH]2)CCN(c2ccccc2)C3)CC1
3,3,C=CC1(Oc2nc(N3CCN(CC)CC3)c3cnc(-c4ccccc4NC)c(F...
4,4,C=CNC(C)CCOc1nc(CN2CCNCC2)c2cnc(-c3ccccc3C(F)(...


In [38]:
full_df = processed_stoned.compute()

full_df.shape

(1010510, 2)

In [39]:
from sklearn.model_selection import train_test_split

In [40]:
train_df, temp_df = train_test_split(
    full_df,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(808408, 2)
(101051, 2)
(101051, 2)


In [41]:
train_df.to_parquet(
    os.path.join(
        BENCHMARK_DIR,
        "train.parquet"
    ),
    index=False
)

val_df.to_parquet(
    os.path.join(
        BENCHMARK_DIR,
        "validation.parquet"
    ),
    index=False
)

test_df.to_parquet(
    os.path.join(
        BENCHMARK_DIR,
        "test.parquet"
    ),
    index=False
)

In [43]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR)
    )

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC_DIR =", SRC_DIR)

PROJECT_ROOT = d:\QCBM Molecular Augmentation Lite
SRC_DIR = d:\QCBM Molecular Augmentation Lite\src


In [45]:
from molaug_qcbm.preprocessing.fingerprints import (
    batch_fingerprints,
    average_diversity
)

print("Fingerprint utilities loaded.")

Fingerprint utilities loaded.


In [47]:
import builtins

In [48]:
descriptor_sample = train_df.sample(
    n=builtins.min(
        10000,
        len(train_df)
    ),
    random_state=42
)

print(
    "Sample Size:",
    len(descriptor_sample)
)

Sample Size: 10000


In [49]:
descriptor_sample[
    ["smiles"]
].head()

,smiles
954558,CC(C)c1ccccc1N1Cc2cnc(OCC3CCCNN3C)nc2C1
392394,CCCC1CCCN(c2nc(OCC3CCCN3)nc3c2CCN(c2cncc4cnc(F...
585151,CCc1c(CN2CC(=O)C(C)(CO)C2)nc(CCc2ccccc2)n1C
120710,CCN(Cc1nc(OCC2CCCN2)[nH]c1N)c1ccccc1CNC
835112,CCCN1CCCC1Cc1nc(N2CNC(CC)NC2C)c2cnc(-c3cccc(C)...


In [50]:
sample_smiles = (
    descriptor_sample["smiles"]
    .dropna()
    .astype(str)
    .tolist()
)

In [51]:
fingerprint_matrix, valid_smiles = (
    batch_fingerprints(
        sample_smiles,
        radius=2,
        n_bits=2048
    )
)

print(
    "Fingerprint Matrix Shape:",
    fingerprint_matrix.shape
)

[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerator
[11:20:44] DEPRECATION WARNING: please use MorganGenerat

Fingerprint Matrix Shape: (9999, 2048)


In [52]:
fingerprint_matrix[:5]

array([[0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5, 2048))

In [53]:
diversity = average_diversity(
    valid_smiles[:1000]
)

print(
    "Sample Diversity:",
    diversity
)

[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerator
[11:21:11] DEPRECATION WARNING: please use MorganGenerat

Sample Diversity: 0.8239815994214089


# Notebook Summary

## Objective

This notebook serves as the data engineering and preprocessing entry point for the QCBM Molecular Augmentation framework.

The goal is to transform large-scale molecular datasets into efficient, analysis-ready formats that can be consumed by:

- Classical Born Machines
- Markov Chain Born Machines
- STONED
- GAN
- VAE
- QGAN
- QVAE
- KMeans + QCBM
- Chow-Liu + QCBM
- GNN + QCBM
- Latent Space + QCBM
- RL + QCBM

---

## Datasets Processed

### 1. STONED Molecular Dataset

Source:

```text
1Mstoned_vsc_initial_dataset_insilico_chemistry42_filtered.csv
```

Contains:

```text
~1.01 Million Molecules
```

Columns:

```text
id
smiles
```

Purpose:

```text
Primary large-scale augmentation dataset.
```

---

### 2. Known KRAS Inhibitors Dataset

Source:

```text
650known_inhibitors_smiles_and_selfies.csv
```

Contains:

```text
636 experimentally validated inhibitors
```

Columns:

```text
id
smiles
selfies
KRAS activity measurements
```

Purpose:

```text
Training and benchmarking molecular generation quality.
```

---

### 3. KRAS Activity Dataset

Source:

```text
KRAS_G12D_inhibitors_update202209_updated.csv
```

Contains:

```text
645 inhibitor molecules
```

Purpose:

```text
Biological validation and benchmarking.
```

---

## Processing Steps Performed

### Step 1

Loaded all datasets using Dask and Pandas.

---

### Step 2

Validated schema and column consistency.

---

### Step 3

Removed missing SMILES records.

---

### Step 4

Removed duplicate molecules.

---

### Step 5

Converted CSV files into Parquet format.

Benefits:

- Faster loading
- Lower storage footprint
- Better scalability
- Native Dask support

---

### Step 6

Created benchmark train/validation/test splits.

Split strategy:

```text
Train      : 80%
Validation : 10%
Test       : 10%
```

---

### Step 7

Prepared molecular fingerprint sampling workflow.

Generated:

- Morgan Fingerprints (ECFP4)
- Tanimoto Similarity utilities
- Diversity estimation workflow

using:

```python
molaug_qcbm.preprocessing.fingerprints
```

---

## Output Artifacts

### Processed Data

```text
data/processed/

├── stoned_1m.parquet
├── known_inhibitors.parquet
├── kras_activity.parquet
├── dataset_summary.csv
```

### Benchmark Splits

```text
data/benchmark_splits/

├── train.parquet
├── validation.parquet
├── test.parquet
```

---

## Why Dask Instead of Spark?

Dataset size:

```text
1,010,510 molecules
```

Dask provides:

- Lower memory overhead
- Simpler deployment
- Better RDKit interoperability
- Native Pandas compatibility
- Faster experimentation on a single workstation

For the current project scale, Dask is more efficient than PySpark.

---

## Next Notebook

```text
01_data_exploration_and_molecular_statistics.ipynb
```

Planned tasks:

- Molecular statistics
- Descriptor distributions
- Fingerprint diversity
- Scaffold analysis
- Dataset visualization
- Benchmark dataset characterization

This notebook establishes the canonical data layer used throughout the entire QCBM Molecular Augmentation framework.